In [2]:
import os
import pandas as pd
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim = mean_spf_trim.set_index('DATE')
forcasters = np.count_nonzero(pd.unique(ind_spf_trim['ID']))
ind_spf_trim = ind_spf_trim.set_index(['DATE', 'ID'])

# __I. Summary Statistics__

In [6]:
mean_stats = pd.DataFrame()
mean_stats.index.name = 'Horizon'
ind_stats = pd.DataFrame()
ind_stats.index.name = 'Horizon'

mean_spf_trim = mean_spf_trim['1981-12-31':'2022-12-31']  # Filter data
ind_spf_trim = ind_spf_trim['1981-12-31':'2022-12-31']

def sum_stats(df, v):
    for j in range(4):
        v.loc[j, 'mean_revisions'] = np.mean(df[f'r_t{j}'] * 100)
        v.loc[j, 'std_revisions'] = np.std(df[f'r_t{j}'] * 100)
        upward = 0
        downward = 0
        na = 0
        total = 0
        for i in df.index:
            if pd.notna(df.loc[i, f'r_t{j}']):
                if df.loc[i, f'r_t{j}'] > 0:
                    upward += 1
                    total += 1
                elif df.loc[i, f'r_t{j}'] < 0:
                    downward += 1
                    total += 1
                else:
                    na += 1
                    total += 1
                v.loc[j, 'revisions_up'] = upward/total
                v.loc[j, 'revisions_downn'] = downward/total
                v.loc[j, 'revisions_na'] = na/total
    
    for j in range(5):
        v.loc[j, 'mean_errors'] = np.mean(df[f'e_t{j}'] * 100)
        v.loc[j, 'std_errors'] = np.std(df[f'e_t{j}'] * 100)

sum_stats(mean_spf_trim, mean_stats)
sum_stats(ind_spf_trim, ind_stats)

print(mean_stats)
print("")
print(ind_stats)

         mean_revisions  std_revisions  revisions_up  revisions_downn  \
Horizon                                                                 
0             -0.009710       0.275037      0.472727         0.527273   
1             -0.031824       0.366757      0.460606         0.539394   
2             -0.052198       0.426467      0.430303         0.569697   
3             -0.072683       0.487412      0.412121         0.587879   
4                   NaN            NaN           NaN              NaN   

         revisions_na  mean_errors  std_errors  
Horizon                                         
0                 0.0     0.005827    0.328570  
1                 0.0     0.003214    0.726011  
2                 0.0    -0.016684    1.057679  
3                 0.0    -0.054147    1.376672  
4                 NaN    -0.112321    1.706823  

         mean_revisions  std_revisions  revisions_up  revisions_downn  \
Horizon                                                                

In [8]:
# Average number of forecasts made by each forecaster
avg_indfor = np.mean(ind_spf_trim.groupby(level='ID').size())

print(avg_indfor)

22.44939271255061


# __II. CSV Export (Optional)__

In [9]:
mean_stats.to_csv('output/mean_stats.csv', sep=',', index=True)
ind_stats.to_csv('output/ind_stats.csv', sep=',', index=True)